# Order of accuracy of finite-difference first-derivative formulae

This notebook loads `fd_results.h5`, produced by `finite_difference.cpp`, and demonstrates that:

- the **forward** difference $f'(x) \approx \dfrac{f(x+h)-f(x)}{h}$ is **first-order** accurate, $E(h) = O(h)$
- the **backward** difference $f'(x) \approx \dfrac{f(x)-f(x-h)}{h}$ is **first-order** accurate, $E(h) = O(h)$
- the **central** difference $f'(x) \approx \dfrac{f(x+h)-f(x-h)}{2h}$ is **second-order** accurate, $E(h) = O(h^2)$

We do this by plotting the absolute error $E(h)$ against the step size $h$ on log-log axes (where an $O(h^p)$ error shows up as a straight line of slope $p$), and by fitting that slope numerically.

**Before running this notebook**, build and run the C++ program so that `fd_results.h5` exists in this directory:
```bash
h5c++ -O2 -std=c++17 -o fd_convergence finite_difference.cpp
./fd_convergence
```
(or use the provided `Makefile` / `CMakeLists.txt`). This requires the HDF5 C++ library (`libhdf5-dev` on Debian/Ubuntu, `hdf5` via Homebrew on macOS).

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

H5_PATH = "fd_results.h5"
f = h5py.File(H5_PATH, "r")

print("Top-level groups:", list(f.keys()))
print()
print("Metadata:")
for k, v in f["metadata"].attrs.items():
    print(f"  {k}: {v}")

In [ ]:
# Test cases stored in the file (excluding the metadata group)
case_names = [k for k in f.keys() if k != "metadata"]
print("Test cases found:", case_names)

for name in case_names:
    g = f[name]
    print(f"\n[{name}]  f(x) = {g.attrs['function_name']},  x0 = {g.attrs['x0']},  "
          f"exact f'(x0) = {g.attrs['exact_derivative']:.10f}")
    print(f"  datasets: {list(g.keys())}, each of length {g['h'].shape[0]}")

## Helper: estimate the observed order of accuracy

For a well-behaved truncation-error-dominated regime, $E(h) \approx C h^p$, so $\log E = \log C + p \log h$.
We estimate $p$ two ways:

1. **Pairwise (local) order**: $p_n = \dfrac{\log(E_n / E_{n+1})}{\log(h_n / h_{n+1})}$ between successive refinements.
2. **Global least-squares slope**: a straight-line fit of $\log E$ vs $\log h$ over a chosen range of $h$.

Very small $h$ eventually brings floating-point round-off error into play (subtractive cancellation in $f(x+h)-f(x)$), which makes the error curve turn back *upward*. We restrict the least-squares fit to a range of $h$ where truncation error dominates, and show the full curve (including the round-off-dominated tail) in the plots so that behavior is visible too.

In [ ]:
def pairwise_orders(h, err):
    h = np.asarray(h); err = np.asarray(err)
    return np.log(err[:-1] / err[1:]) / np.log(h[:-1] / h[1:])

def fit_slope(h, err, h_min, h_max):
    h = np.asarray(h); err = np.asarray(err)
    mask = (h >= h_min) & (h <= h_max) & (err > 0)
    log_h = np.log(h[mask])
    log_e = np.log(err[mask])
    p, log_C = np.polyfit(log_h, log_e, 1)
    return p, log_C, mask

## Convergence plots

For each test function we plot $|E(h)|$ vs $h$ on log-log axes for all three schemes, together with reference lines proportional to $h^1$ and $h^2$ for comparison. The fitted slope in each legend entry is the observed order of accuracy: it should come out close to **1** for forward/backward and close to **2** for central, over the range where truncation error dominates (roughly $10^{-6} \lesssim h \lesssim 10^{-2}$ here).

In [ ]:
FIT_H_MIN, FIT_H_MAX = 1e-6, 1e-2  # range where truncation error dominates round-off

schemes = [
    ("error_forward",  "Forward  (expected O(h))",   "tab:blue"),
    ("error_backward", "Backward (expected O(h))",   "tab:orange"),
    ("error_central",  "Central  (expected O(h^2))", "tab:green"),
]

results_summary = []

for name in case_names:
    g = f[name]
    h = g["h"][:]
    func_label = g.attrs["function_name"]
    x0 = g.attrs["x0"]

    fig, ax = plt.subplots(figsize=(7, 5.5))

    for dset_name, label, color in schemes:
        err = g[dset_name][:]
        p, log_C, mask = fit_slope(h, err, FIT_H_MIN, FIT_H_MAX)
        results_summary.append({"function": func_label, "scheme": label, "order": p})

        ax.loglog(h, err, "o-", ms=3, color=color, label=f"{label}: fitted order = {p:.2f}")
        ax.loglog(h[mask], err[mask], "o", ms=5, color=color, mfc="none", mew=1.5)

    # Reference slope lines anchored near the middle of the fit range
    h_ref = np.array([FIT_H_MIN, FIT_H_MAX])
    anchor_idx = np.argmin(np.abs(h - 1e-4))
    e_anchor = g["error_forward"][anchor_idx]
    ax.loglog(h_ref, e_anchor * (h_ref / h[anchor_idx]) ** 1, "k--", lw=1, alpha=0.6, label="slope 1 reference")
    ax.loglog(h_ref, e_anchor * (h_ref / h[anchor_idx]) ** 2, "k:",  lw=1, alpha=0.6, label="slope 2 reference")

    ax.set_xlabel("step size $h$")
    ax.set_ylabel("absolute error $|E(h)|$")
    ax.set_title(f"Finite-difference convergence: $f(x)$ = {func_label}, $x_0$ = {x0}")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Summary table of fitted orders

Across every test function, the forward and backward schemes should fit to an order close to **1**, and the central scheme close to **2**, confirming their theoretical accuracy.

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(results_summary)
    df["order"] = df["order"].round(3)
    display(df.pivot(index="function", columns="scheme", values="order"))
except ImportError:
    for row in results_summary:
        print(f"{row['function']:12s} | {row['scheme']:28s} | order = {row['order']:.3f}")

## Local (pairwise) order of accuracy

As a second, complementary check, we look at the *local* order estimated from each successive pair of step sizes for a single test case. This shows the order settling in toward 1 (forward/backward) and 2 (central) as $h \to 0$, before round-off error takes over at the very smallest $h$.

In [ ]:
name = "sin"  # try "exp" or "cubic" as well
g = f[name]
h = g["h"][:]

print(f"{'h':>14s}  {'p_forward':>10s}  {'p_backward':>11s}  {'p_central':>10s}")
p_fwd = pairwise_orders(h, g["error_forward"][:])
p_bwd = pairwise_orders(h, g["error_backward"][:])
p_ctr = pairwise_orders(h, g["error_central"][:])
for i in range(len(p_fwd)):
    print(f"{h[i]:14.3e}  {p_fwd[i]:10.3f}  {p_bwd[i]:11.3f}  {p_ctr[i]:10.3f}")

## Conclusion

- The **forward** and **backward** difference formulae have error that shrinks linearly with $h$ (slope $\approx 1$ on the log-log plot): halving $h$ roughly halves the error. This confirms they are **first-order accurate**.
- The **central** difference formula has error that shrinks quadratically with $h$ (slope $\approx 2$): halving $h$ roughly quarters the error. This confirms it is **second-order accurate**.
- At very small $h$ (below roughly $10^{-7}$–$10^{-8}$ for double precision here), floating-point round-off error in evaluating $f(x \pm h) - f(x)$ starts to dominate, and the error curves turn upward instead of continuing to decrease — a classic truncation-error-vs-round-off-error trade-off that sets a practical lower bound on how small a useful step size can be.

In [ ]:
f.close()